**Table of contents**<a id='toc0_'></a>    
1. [Data Sources](#toc1_)    
2. [Process datasets](#toc2_)    
2.1. [Create LSOA -> Ward mapping table](#toc2_1_)    
2.2. [Household composition](#toc2_2_)    
2.2.1. [Standardise categories, map LSOAs to wards, and merge all census years into one table](#toc2_2_1_)    
2.3. [Household size](#toc2_3_)    
2.3.1. [Map LSOAs to wards, and merge all census years into one table](#toc2_3_1_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=true
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

<h1>HDRC055 - Household Composition</h1>

**Author:** Yanhua Xu

In [1]:
import pandas as pd
from pathlib import Path
from rich import print

data_dir = Path("./data")
raw_data_dir = data_dir / "raw"

::: {.callout-important title="Data processing summary" collapse="false"}

This notebook processes the manually downloaded Nomis data for **Household composition** and **Household size**.

The input data currently include records at **LSOA level**, **local authority / district level (Bradford)**, and **England level**.

For **Household composition**, the workflow includes:

- standardising selected column names across census years;
- identifying LSOA-level records and mapping them to **2025 ward** geography;
- recalculating ward-level household counts using weighted allocation;
- keeping non-LSOA records, such as district-level and England-level records, unchanged;
- standardising geography type labels: replace `local authorities: district / unitary (as of ...)` with `local authority district"`;
- harmonising household composition categories across census years;
- splitting multi-level household composition categories into `main_category` and corresponding `sub_category` columns;
- calculating the percentage share of each category relative to the total number of households in the same geography and census year.

When an LSOA is split across multiple 2025 wards, the household counts are allocated to the corresponding wards using the split weights from the lookup table. For example, if one LSOA is split equally across three wards, then one third of the household count for each category is assigned to each ward. The allocated values are then summed to produce the recalculated ward-level estimates.

For **Household size**, the workflow follows the same general process: column names are standardised, LSOA-level records are mapped to 2025 wards, weighted ward-level counts are recalculated, and category percentages are calculated. However, household size categories are already broadly consistent across census years, so no additional category harmonisation is applied.

The only category removed from the household size data is the category **"0 people in household"**, because it contains zero households and is not needed for the final dataset.

:::

# 1. <a id='toc1_'></a>[Data Sources](#toc0_)

Data can be downloaded from the [Nomis website](https://www.nomisweb.co.uk/). Below are links to the relevant datasets for each census year.

| Theme | 2001 | 2011 | 2021 |
|---|---|---|---|
| Household composition | [UV065](https://www.nomisweb.co.uk/datasets/uv065)<br>Geography: LSOA, MSOA, 2003 CAS Ward <br> Units: Households <br> # categories: 26 | [KS105EW](https://www.nomisweb.co.uk/census/2011/ks105ew)<br>Geography: LSOA, MSOA, 2011 Ward <br> Units: Households <br> # categories: 22 <br> Note: [QS113EW](https://www.nomisweb.co.uk/census/2011/qs113ew) has similar categories than 2001 version | [TS003](https://www.nomisweb.co.uk/datasets/c2021ts003)<br>Geography: LSOA, MSOA, 2022 Ward <br> Units: Households <br> # categories: 22 |
| Household size | [UV051](https://www.nomisweb.co.uk/datasets/uv051)<br>Geography: LSOA, MSOA, 2003 CAS Ward <br> Unit: Persons <br> # categories: 9 | [QS406EW](https://www.nomisweb.co.uk/census/2011/qs406ew)<br>Geography: LSOA, MSOA, 2011 Ward <br> Unit: Household spaces <br> # categories: 9 | [TS017](https://www.nomisweb.co.uk/datasets/c2021ts017)<br>Geography: LSOA, MSOA, 2022 Ward <br> Unit: Household <br> # categories: 10 |
| Household composition by ethnicity of HRP | [ST106](https://www.nomisweb.co.uk/datasets/st106)<br>Geography: 2003 Standard Table Ward <br> Units: Households <br> # household composition categories: 26 <br> # ethnic group categories: 22 | [LC1201EW](https://www.nomisweb.co.uk/census/2011/lc1201ew)<br>Geography: LSOA, MSOA, 2011 Ward <br> Units: Households <br> # household composition categories: 7 <br> # ethnic group categories: 9 <br> <br>[DC1201EW](https://www.nomisweb.co.uk/census/2011/dc1201ew) <br> Geography: 2011 census merged wards; MSOA <br> Units: Households <br> # household composition categories: 22 <br> # ethnic group categories: 24 | [RM058](https://www.nomisweb.co.uk/datasets/c2021rm058)<br>Geography: LSOA, MSOA, 2022 Ward <br> Units: Persons <br> # household composition categories: 7 <br> # ethnic group categories: 9 |
| Household composition by age | Not ideal / use [TT008](https://www.nomisweb.co.uk/datasets/tt008) if needed<br>Geography: 2003 Standard Table Ward | [LC1109EW](https://www.nomisweb.co.uk/census/2011/lc1109ew)<br>Geography: LSOA, MSOA, 2011 Ward <br> Units: Persons <br> # age categories: 6 <br> # household composition categories: 20 | [RM057](https://www.nomisweb.co.uk/datasets/c2021rm057)<br>Geography: MSOA, 2022 Ward <br> Units: Persons <br> # age categories: 6 <br> # household composition categories: 21 |
| Gender identity by family composition | N/A | N/A | [RM166](https://www.nomisweb.co.uk/datasets/c2021rm166)<br>Geography: MSOA, 2022 Ward |
| Theme table / wider household variables | [TT008](https://www.nomisweb.co.uk/datasets/tt008)<br>Geography: 2003 Standard Table Ward | N/A | N/A |

**Key parameters**:
- Geography: Super Output Areas - Lower Layer (Bradford); countries (England); local authorities: district / unitary (Bradford)
- Data Format: Nomis API

**LSOA to Ward lookup table**:

To map LSOA-level data to Ward-level data, we also need following lookup tables:

The 2001 LSOA to 2011 LSOA lookup table can be downloaded from [ONS](https://geoportal.statistics.gov.uk/search?q=Lower%20Layer%20Super%20Output%20Area%20(2001)%20to%20Lower%20Layer%20Super%20Output%20Area%20(2011)%20to%20Local%20Authority%20District%20(2011)%20Lookup)

The 2011 LSOA to 2021 LSOA lookup table can be downloaded from [ONS](https://geoportal.statistics.gov.uk/datasets/ons::lsoa-2011-to-lsoa-2021-to-local-authority-district-2022-exact-fit-lookup-for-ew-v3/about)

The 2021 LSOA to 2025 Ward lookup table can be downloaded from [data.gov.uk](https://www.data.gov.uk/dataset/8fd0e85c-3f46-4851-a041-e105b3b32315/lsoa-2021-to-electoral-ward-2025-to-lad-2025-best-fit-lookup-in-ew-v21)

It seems possible to create a custom dataset or ask ONS to produce one: [Create a custom dataset](https://www.ons.gov.uk/datasets/create)

<!-- 2001:
[Univariate tables](https://www.nomisweb.co.uk/sources/census_2001_uv)
[Standard Tables](https://www.nomisweb.co.uk/sources/census_2001_st)
[Census 2001 Tables](https://www.nomisweb.co.uk/census/2001/data_finder)
[UV065 Household composition (households)](https://www.nomisweb.co.uk/census/2001/uv065)
[ST051 Tenure and household size by number of rooms](https://www.nomisweb.co.uk/census/2001/st051) -->

<!-- 2011:

[Census 2011 Tables](https://www.nomisweb.co.uk/census/2011/data_finder)

2021:

[Ready Made Tables](https://www.nomisweb.co.uk/sources/census_2021_rm) -->


> The unit of measure differs slightly across Census years for household size tables (e.g. households vs household spaces). However, the category structure and overall data ranges appear broadly comparable across 2001, 2011 and 2021, suggesting these tables are likely measuring similar household-size distributions.

::: {.callout-note title="Note on current data coverage" collapse="false"}

The current harmonised dataset only includes **Household composition** and **Household size**.

Breakdowns by **ethnicity of the Household Reference Person (HRP)**, **age**, **sex**, or other demographic characteristics have not yet been incorporated. If these breakdowns are added later, extra care will be needed because the available datasets are not fully consistent across census years.

In particular, note that:

- available geographies may differ by year;
- category definitions may change over time;
- some tables are only available at less detailed geographic levels;
- some cross-tabulated tables use broader categories than the main household composition tables;
- it may not always be possible to produce highly detailed, directly comparable breakdowns for 2001, 2011, and 2021.

As a result, any future demographic breakdowns may require additional harmonisation and may need to use broader categories to maintain comparability across census years.

:::

::: {.callout-note title="Note on using the Nomis API" collapse="false"}

Currently we manually download the data from the Nomis website, but it is possible to fetch data directly using the Nomis API. A typical API request follows this structure:

`https://www.nomisweb.co.uk/api/v01/dataset/{DATASET_ID}.data.csv?&date={DATE}geography={GEOGRAPHY_CODE}&measures={MEASURE_CODE}`

For example:

`https://www.nomisweb.co.uk/api/v01/dataset/NM_1681_1.data.csv?&date=latest&geography=2092957699&c_hhchuk11=0...25&measures=20100`

The exact API parameters vary by dataset, so the metadata and codelists should be checked before building each query. In this example:

- `NM_1681_1` is the Nomis dataset ID for `UV065 - Household composition`;
- `.data.csv` returns the data in CSV format;
- `date=latest` returns the latest available period for the dataset;
- `geography=2092957699` specifies the selected geography, `2092957699` is for England here;
- `c_hhchuk11=0...25` requests household composition categories from code `0` to `25`;
- `measures=20100` requests the count measure.

The exact API parameters vary by dataset, so the metadata and codelists should be checked before building each query. Common parameters include:

- `date`: the census year or time period, such as `latest`;
- `geography`: the geography code or geography type to return;
- table-specific category parameters, such as `c_hhchuk11` for household composition in this dataset;
- `measures`: the measure to return, such as counts, percentages, or rates.

When using the API, note that dataset IDs, category parameter names, category codes, geography codes, and measure codes can differ across datasets and census years. For example, the household composition category parameter may not always be called `c_hhchuk11`, and the range of category codes may not always be `0...25`.

:::

In [2]:
print("[bold]\nThe project uses the following data folders:[/bold]\n")

print(f"Raw data folder: [bold red]{raw_data_dir}[/bold red]")
print(f"Processed data folder: [bold red]{data_dir}[/bold red]")

The project uses the following data folders:

Raw data folder: data/raw

Processed data folder: data

# 2. <a id='toc2_'></a>[Process datasets](#toc0_)

## 2.1. <a id='toc2_1_'></a>[Create LSOA -> Ward mapping table](#toc0_)

In [3]:
lookup_tables = {
    '01OA-11OA': {
        'filename': 'Lower_Layer_Super_Output_Area_(2001)_to_Lower_Layer_Super_Output_Area_(2011)_to_Local_Authority_District_(2011)_Lookup_in_England_and_Wales.csv',
        'columns': ['LSOA01CD', 'LSOA11CD'],
    },
    '11OA-21OA': {
        'filename': 'LSOA_(2011)_to_LSOA_(2021)_to_Local_Authority_District_(2022)_Exact_Fit_Lookup_for_EW_(V3).csv',
        'columns': ['LSOA11CD', 'LSOA21CD'],
    },
    # '21OA-24WD': {
    #     'filename': 'LSOA_(2021)_to_Electoral_Ward_(2024)_to_LAD_(2024)_Best_Fit_Lookup_in_EW.csv',
    #     'columns': ['LSOA21CD', 'WD24NM'],
    # },
    '21OA-25WD': {
        'filename': 'LSOA_(2021)_to_Electoral_Ward_(2025)_to_LAD_(2025)_Best_Fit_Lookup_in_EW_v2.csv',
        'columns': ['LSOA21CD', 'WD25CD', 'WD25NM'],
    }
}

dfs_lookup = {
    name: pd.read_csv(
        raw_data_dir / info['filename'],
        usecols=info['columns']
    ).drop_duplicates()
    for name, info in lookup_tables.items()
}

In [4]:
# create lsoa - 2021 ward lookup table 
ward_mapping = (
    dfs_lookup['01OA-11OA'][['LSOA01CD', 'LSOA11CD']]
    .merge(dfs_lookup['11OA-21OA'], on='LSOA11CD', how='left')
    .merge(dfs_lookup['21OA-25WD'], on='LSOA21CD', how='left')
)

ward_mapping = ward_mapping.drop_duplicates()

Here is the structure of mapping table for LSOA to Ward for each census year:

In [5]:
ward_mapping.head()

,LSOA01CD,LSOA11CD,LSOA21CD,WD25CD,WD25NM
0,E01031349,E01031349,E01031349,E05007565,Eastbrook
1,E01031350,E01031350,E01031350,E05007566,Hillside
2,E01031351,E01031351,E01031351,E05007566,Hillside
3,E01031352,E01031352,E01031352,E05007566,Hillside
4,E01031370,E01031370,E01031370,E05007573,Southlands


In [6]:
def lsoa_to_ward(df, year, lsoa_col, ward_col, ward_mapping):
    df = df.copy()
    ward_mapping = ward_mapping.copy()
    year = str(year)

    lsoa_cd_col = {
        "2001": "LSOA01CD",
        "2011": "LSOA11CD",
        "2021": "LSOA21CD"
    }

    source_lsoa_col = lsoa_cd_col[year]

    # clean LSOA codes
    df[lsoa_col] = df[lsoa_col].astype(str).str.strip()
    ward_mapping[source_lsoa_col] = ward_mapping[source_lsoa_col].astype(str).str.strip()

    lookup_table = (
        ward_mapping
        .loc[
            ward_mapping[source_lsoa_col].isin(df[lsoa_col]),
            [source_lsoa_col, "WD25CD", "WD25NM"]
        ]
        .drop_duplicates()
    )

    # identify one-to-many LSOA -> ward cases
    one_to_many = (
        lookup_table
        .loc[lookup_table.duplicated(subset=source_lsoa_col, keep=False)]
        .sort_values(source_lsoa_col)
    )

    if not one_to_many.empty:
        print(
            f"Warning: {one_to_many[source_lsoa_col].nunique()} {source_lsoa_col} values "
            f"map to multiple WD25NM values. These will be split equally using weights."
        )
        display(one_to_many)

    # add equal-split weight
    lookup_table["n_wards"] = (
        lookup_table
        .groupby(source_lsoa_col)["WD25NM"]
        .transform("nunique")
    )

    lookup_table["weight"] = 1 / lookup_table["n_wards"]

    # merge instead of map, because one LSOA can map to multiple wards
    df_out = df.merge(
        lookup_table[[source_lsoa_col, "WD25CD", "WD25NM", "weight"]],
        left_on=lsoa_col,
        right_on=source_lsoa_col,
        how="left"
    )

    # rename ward column if needed
    df_out = df_out.rename(columns={"WD25NM": ward_col})

    # drop duplicated source LSOA column if it is different from df lsoa_col
    if source_lsoa_col != lsoa_col and source_lsoa_col in df_out.columns:
        df_out = df_out.drop(columns=[source_lsoa_col])

    return df_out

## 2.2. <a id='toc2_2_'></a>[Household composition](#toc0_)

We manually downloaded the data from Nomis website, and first have a look at what files we have for household composition:

In [7]:
all_files = [f for f in raw_data_dir.iterdir() if f.is_file()]

comp_files = sorted([f for f in all_files if 'composition' in str(f)])
print(comp_files)

[
    PosixPath('data/raw/2001 UV065 - Household composition (households).csv'),
    PosixPath('data/raw/2011 KS105EW - Household composition.csv'),
    PosixPath('data/raw/2021 TS003 - Household composition.csv')
]

Have a look at the data sturcture:

In [8]:
pd.read_csv(comp_files[0]).head()

,DATE,DATE_NAME,DATE_CODE,DATE_TYPE,DATE_TYPECODE,DATE_SORTORDER,GEOGRAPHY,GEOGRAPHY_NAME,GEOGRAPHY_CODE,GEOGRAPHY_TYPE,...,MEASURES,MEASURES_NAME,OBS_VALUE,OBS_STATUS,OBS_STATUS_NAME,OBS_CONF,OBS_CONF_NAME,URN,RECORD_OFFSET,RECORD_COUNT
0,2001,2001,2001,date,0,0,1946157124,Bradford,E08000032,local authorities: district / unitary (prior t...,...,20100,Value,180246,A,Normal Value,F,Free (free for publication),Nm-1681d1d32016e0d1946157124d0d20100,0,8034
1,2001,2001,2001,date,0,0,1946157124,Bradford,E08000032,local authorities: district / unitary (prior t...,...,20100,Value,51991,A,Normal Value,F,Free (free for publication),Nm-1681d1d32016e0d1946157124d1d20100,1,8034
2,2001,2001,2001,date,0,0,1946157124,Bradford,E08000032,local authorities: district / unitary (prior t...,...,20100,Value,25856,A,Normal Value,F,Free (free for publication),Nm-1681d1d32016e0d1946157124d2d20100,2,8034
3,2001,2001,2001,date,0,0,1946157124,Bradford,E08000032,local authorities: district / unitary (prior t...,...,20100,Value,26135,A,Normal Value,F,Free (free for publication),Nm-1681d1d32016e0d1946157124d3d20100,3,8034
4,2001,2001,2001,date,0,0,1946157124,Bradford,E08000032,local authorities: district / unitary (prior t...,...,20100,Value,114257,A,Normal Value,F,Free (free for publication),Nm-1681d1d32016e0d1946157124d4d20100,4,8034


In [9]:
col_mappings = {
    "DATE": "census",
    "GEOGRAPHY_CODE": "geography_code",
    "GEOGRAPHY_TYPE": "geography_type",
    "GEOGRAPHY_NAME": "geography_name",

    "C_HHCHUK11": "class_order",
    "CELL_SORTORDER": "class_order",
    "C2021_HHCOMP_15_SORTORDER": "class_order",
    
    "C_HHCHUK11_NAME": "class_name",
    "CELL_NAME": "class_name",
    "C2021_HHCOMP_15_NAME": "class_name",
    
    "C2021_HHCOMP_15_TYPE": "class_type",
    "CELL_TYPE": "class_type",
    "C_HHCHUK11_TYPE": "class_type",

    "OBS_VALUE": "number",
}

In [10]:
print("[bold]Only columns listed in the mapping and present in each file will be selected.[/bold]")
print("[bold]The selected columns will be renamed as follows:[/bold]\n")

mapping_df = pd.DataFrame(
    [
        {
            "Original column": old_col,
            "Renamed column": new_col,
        }
        for old_col, new_col in col_mappings.items()
    ]
)

print(mapping_df)

Only columns listed in the mapping and present in each file will be selected.

The selected columns will be renamed as follows:

Original column  Renamed column
0                        DATE          census
1              GEOGRAPHY_CODE  geography_code
2              GEOGRAPHY_TYPE  geography_type
3              GEOGRAPHY_NAME  geography_name
4                  C_HHCHUK11     class_order
5              CELL_SORTORDER     class_order
6   C2021_HHCOMP_15_SORTORDER     class_order
7             C_HHCHUK11_NAME      class_name
8                   CELL_NAME      class_name
9        C2021_HHCOMP_15_NAME      class_name
10       C2021_HHCOMP_15_TYPE      class_type
11                  CELL_TYPE      class_type
12            C_HHCHUK11_TYPE      class_type
13                  OBS_VALUE          number

In [11]:
keep_cols = list(dict.fromkeys(col_mappings.values()))

dfs_comp = {
    f.name[:4]: (
        pd.read_csv(f)
        .rename(columns=col_mappings)
        .loc[:, lambda df: [c for c in keep_cols if c in df.columns]]
    )
    for f in comp_files
}

### 2.2.1. <a id='toc2_2_1_'></a>[Standardise categories, map LSOAs to wards, and merge all census years into one table](#toc0_)

::: {.callout-note title="Harmonisation note" collapse="false"}

By default, the harmonised categories use the 2021 Census household composition categories as the reference classification. Categories from 2001 and 2011 are mapped onto the closest equivalent 2021 category wherever possible.

Categories are merged when earlier censuses provide a more detailed breakdown than 2021. For example, where 2001 distinguishes between households with one dependent child and households with two or more dependent children, these categories are combined into the broader 2021 category "with dependent children". Similarly, older "other household" subcategories, such as all-student, all-pensioner, and other households, are merged where 2021 reports them as a combined category.

A harmonised category is renamed or differs from the exact 2021 wording where the 2021 label does not fully represent the earlier census definitions, or where a broader comparable category is needed across all years. This mainly applies to categories affected by definitional changes, such as "pensioner", "aged 65 and over", and "aged 66 years and over", or categories where civil partnerships were introduced into later census classifications.

:::

In [12]:
# load class mappings
class_mapping_file = data_dir / "household_composition_class_mapping.xlsx"
class_mapping = pd.read_excel(class_mapping_file).dropna(how='all')
class_mapping = class_mapping.rename(columns=str)

print(
    "[bold]The household composition categories will be harmonised using the mapping file saved at:[/bold] "
    f"[bold red]{class_mapping_file}[/bold red]"
)
class_mapping

The household composition categories will be harmonised using the mapping file saved at: 
data/household_composition_class_mapping.xlsx

,Harmonised category,2001,2011,2021
0,Total: All households,All categories: Household composition,All categories: Household composition,Total: All households
2,One person household,One person,One person household,One-person household
3,One person household: Older person,One person: Pensioner,One person household: Aged 65 and over,One-person household: Aged 66 years and over
4,One person household: Other,One person: Other,One person household: Other,One-person household: Other
6,Single family household,One family and no others,One family household,Single family household
7,Single family household: Older person,One family and no others: All pensioner,One family only: All aged 65 and over,Single family household: All aged 66 years and...
8,Single family household: Married or civil part...,One family and no others: Married Couple House...,One family only: Married or same-sex civil par...,Single family household: Married or civil part...
9,Single family household: Married or civil part...,One family and no others: Married Couple House...,One family only: Married or same-sex civil par...,Single family household: Married or civil part...
10,Single family household: Married or civil part...,One family and no others: Married Couple House...,One family only: Married or same-sex civil par...,Single family household: Married or civil part...
11,Single family household: Married or civil part...,One family and no others: Married Couple House...,NaN,NaN


The next step harmonises the household composition data and maps LSOA-level records to 2025 ward-level geography.

For each census year, LSOA-level records are identified and mapped to 2025 wards using the LSOA-to-ward lookup. Non-LSOA records (district or England level) are kept unchanged. Where an LSOA is split across more than one ward, the household count is apportioned using the corresponding ward weight. For example, if one LSOA is split equally across three wards, then one third of the household count for each category is assigned to each ward. The allocated values are then summed to produce the recalculated ward-level estimates.


Household composition categories are then harmonised across census years using the category mapping table. The harmonised category is also split into a hierarchical structure: `main_category` stores the broad household composition group, while `sub_category1`, `sub_category2`, and subsequent columns store more detailed category levels based on the text after each colon.

The percentage share of each category is calculated within each geography and census year by dividing the category count by the total number of households.

In [13]:
dfs_new = []
save_path = data_dir / 'household_composition_ward.xlsx'

for year, df in dfs_comp.items():
    # print(year)
    lsoa_col = "geography_code"
    ward_col = "ward"
    geo_name = 'geography_name'
    num_col = "number"
    cat_col = "class_name"
    
    # get lsoa-level data
    is_lsoa = df["geography_type"].str.contains("lower layer", case=False, na=False)
    df_lsoa = df.loc[is_lsoa].copy()

    df_other = df.loc[~is_lsoa].copy()
    
    # map lsoa to 2021 ward
    df_ward_tmp = lsoa_to_ward(df_lsoa, year, lsoa_col, ward_col, ward_mapping)
    
    df_ward_tmp.drop(columns=['geography_code', geo_name], inplace=True)
    df_ward_tmp.rename(columns={'WD25CD': 'geography_code', 'ward': geo_name}, inplace=True)
    
    df_weight = df_ward_tmp.copy()

    df_weight[f"weighted_{num_col}"] = df_weight[num_col] * df_weight["weight"]

    global_keys = [cat_col, "census"]
    unique_keys = [geo_name] + global_keys

    # ward-level
    df_weight = (
        df_weight
        .groupby(unique_keys, as_index=False)
        .agg(
            **{
                num_col: (f"weighted_{num_col}", "sum"),
            }
        )
    )
    
    df_ward = df_weight.merge(df_ward_tmp.drop(columns=['weight', num_col]).drop_duplicates(), how='left', on=unique_keys)
    df_ward['geography_type'] = '2025 ward'
    
    df_new = pd.concat([df_other, df_ward], ignore_index=True)
    
    # curate categories
    cat_mapping = class_mapping[[year, 'Harmonised category']].dropna().set_index(year)['Harmonised category']
    df_new[cat_col] = df_new[cat_col].str.strip().map(cat_mapping).fillna(df_new[cat_col].str.strip())
    
    dfs_new.append(df_new)

df_comp_ward = pd.concat(dfs_new, ignore_index=True)

main_cat_col = "main_category"
sub_cat_col = "sub_category"
total_row = "Total: All households"
pct_col = "pct"

is_total = df_comp_ward[cat_col].isin([total_row])
parts = df_comp_ward.loc[~is_total, cat_col].str.split(":")
num_categories = parts.str.len().max()
df_comp_ward[main_cat_col] = df_comp_ward[cat_col]
df_comp_ward.loc[~is_total, main_cat_col] = parts.str[0].str.strip()
for i in range(1, num_categories):
    df_comp_ward[f"{sub_cat_col}{i}"] = parts.str[i].str.strip()

totals = df_comp_ward[is_total][[geo_name, num_col, "census"]].rename(columns={num_col: 'total_households'})
df_comp_ward = df_comp_ward.merge(totals, on=[geo_name, "census"], how="left")
df_comp_ward[pct_col] = (df_comp_ward[num_col] / df_comp_ward["total_households"] * 100).round(1)

# standarise geo type
geo_type_col = "geography_type"
is_district = df_comp_ward[geo_type_col].str.contains("district|unitary|local authorit", case=False, na=False)
df_comp_ward.loc[is_district, geo_type_col] = "local authority district"

df_comp_ward.to_excel(save_path, index=False)

print(f"The final data has been saved at: [bold red]{save_path}[/bold red]")

Warning: 1 LSOA01CD values map to multiple WD25NM values. These will be split equally using weights.

,LSOA01CD,WD25CD,WD25NM
3584,E01010823,E05001347,City
3588,E01010823,E05001359,Manningham


The final data has been saved at: data/household_composition_ward.xlsx

Have a look at the structure of our final dataset:

In [14]:
df_comp_ward.head()

,census,geography_code,geography_type,geography_name,class_order,class_name,class_type,number,main_category,sub_category1,sub_category2,total_households,pct
0,2001,E08000032,local authority district,Bradford,0,Total: All households,Household Composition,180246.0,Total: All households,NaN,NaN,180246.0,100.0
1,2001,E08000032,local authority district,Bradford,1,One person household,Household Composition,51991.0,One person household,NaN,NaN,180246.0,28.8
2,2001,E08000032,local authority district,Bradford,2,One person household: Older person,Household Composition,25856.0,One person household,Older person,NaN,180246.0,14.3
3,2001,E08000032,local authority district,Bradford,3,One person household: Other,Household Composition,26135.0,One person household,Other,NaN,180246.0,14.5
4,2001,E08000032,local authority district,Bradford,4,Single family household,Household Composition,114257.0,Single family household,NaN,NaN,180246.0,63.4


In [15]:
wide_save_path = data_dir / "household_composition_ward_wide.xlsx"

id_cols = [
    "geography_code",
    "geography_name",
    "geography_type",
    "census",
]

df_plot = df_comp_ward.loc[
    df_comp_ward["class_name"] != "Total: All households"
].copy()

df_comp_wide_pct = (
    df_plot
    .pivot_table(
        index=id_cols,
        columns="class_name",
        values="pct",
        aggfunc="sum"
    )
    .add_suffix("_pct")
    .reset_index()
)

df_comp_wide_num = (
    df_plot
    .pivot_table(
        index=id_cols,
        columns="class_name",
        values="number",
        aggfunc="sum"
    )
    .add_suffix("_number")
    .reset_index()
)

df_comp_wide = df_comp_wide_pct.merge(
    df_comp_wide_num,
    on=id_cols,
    how="left"
)

df_comp_wide.columns.name = None

df_comp_wide.to_excel(wide_save_path, index=False)

print(f"The wide data has been saved at: [bold red]{wide_save_path}[/bold red]")

The wide data has been saved at: data/household_composition_ward_wide.xlsx

Have a look at the structure of our final dataset (wide format):

In [16]:
df_comp_wide.head()

,geography_code,geography_name,geography_type,census,One person household_pct,One person household: Older person_pct,One person household: Other_pct,Other household types_pct,"Other household types: Other, including all full-time students and all aged 66 years and over_pct",Other household types: With dependent children_pct,...,Single family household: Lone parent family_number,Single family household: Lone parent family: All children non-dependent_number,Single family household: Lone parent family: With dependent children_number,Single family household: Married or civil partnership couple_number,Single family household: Married or civil partnership couple: All children non-dependent_number,Single family household: Married or civil partnership couple: No children_number,Single family household: Married or civil partnership couple: With dependent children_number,Single family household: Older person_number,Single family household: Other single family household_number,Single family household: Other single family household: Other family composition_number
0,E05001341,Baildon,2025 ward,2001,27.5,16.9,10.6,3.5,2.3,1.2,...,501.0,202.0,299.0,2767.0,464.0,1035.0,1268.0,754.0,NaN,NaN
1,E05001341,Baildon,2025 ward,2011,31.3,14.6,16.6,4.4,2.5,1.9,...,489.0,191.0,298.0,2656.0,451.0,1098.0,1107.0,745.0,NaN,NaN
2,E05001341,Baildon,2025 ward,2021,31.8,16.8,15.0,3.3,2.1,1.2,...,642.0,218.0,424.0,2357.0,396.0,966.0,995.0,988.0,32.0,32.0
3,E05001342,Bingley,2025 ward,2001,29.5,16.2,13.3,4.5,2.8,1.7,...,503.0,164.0,339.0,2618.0,438.0,948.0,1232.0,690.0,NaN,NaN
4,E05001342,Bingley,2025 ward,2011,31.8,14.2,17.6,5.2,3.7,1.5,...,631.0,212.0,419.0,2746.0,390.0,1102.0,1254.0,716.0,NaN,NaN


## 2.3. <a id='toc2_3_'></a>[Household size](#toc0_)

We manually downloaded the data from Nomis website, and first have a look at what files we have for household size:

In [17]:
size_files = sorted([f for f in all_files if 'size' in str(f) or 'UV051' in str(f)])
print(size_files)

[
    PosixPath('data/raw/2001 UV051 - Number of people living in households.csv'),
    PosixPath('data/raw/2011 QS406EW - Household size.csv'),
    PosixPath('data/raw/2021 TS017 - Household size.csv')
]

In [18]:
col_mappings = {
    "DATE": "census",
    "GEOGRAPHY_CODE": "geography_code",
    "GEOGRAPHY_TYPE": "geography_type",
    "GEOGRAPHY_NAME": "geography_name",

    "CELL_SORTORDER": "class_order",
    "C2021_HHSIZE_10_SORTORDER": "class_order",
    
    "CELL_NAME": "class_name",
    "C2021_HHSIZE_10_NAME": "class_name",
    
    "CELL_TYPE": "class_type",
    "C2021_HHSIZE_10_TYPE": "class_type",

    "OBS_VALUE": "number",
}

In [19]:
print("[bold]Only columns listed in the mapping and present in each file will be selected.[/bold]")
print("[bold]The selected columns will be renamed as follows:[/bold]\n")

mapping_df = pd.DataFrame(
    [
        {
            "Original column": old_col,
            "Renamed column": new_col,
        }
        for old_col, new_col in col_mappings.items()
    ]
)

print(mapping_df)

Only columns listed in the mapping and present in each file will be selected.

The selected columns will be renamed as follows:

Original column  Renamed column
0                        DATE          census
1              GEOGRAPHY_CODE  geography_code
2              GEOGRAPHY_TYPE  geography_type
3              GEOGRAPHY_NAME  geography_name
4              CELL_SORTORDER     class_order
5   C2021_HHSIZE_10_SORTORDER     class_order
6                   CELL_NAME      class_name
7        C2021_HHSIZE_10_NAME      class_name
8                   CELL_TYPE      class_type
9        C2021_HHSIZE_10_TYPE      class_type
10                  OBS_VALUE          number

In [20]:
keep_cols = list(dict.fromkeys(col_mappings.values()))

dfs_size = {
    f.name[:4]: (
        pd.read_csv(f)
        .rename(columns=col_mappings)
        .loc[:, lambda df: [c for c in keep_cols if c in df.columns]]
    )
    for f in size_files
}

### 2.3.1. <a id='toc2_3_1_'></a>[Map LSOAs to wards, and merge all census years into one table](#toc0_)

In [21]:
dfs_new = []
save_path = data_dir / 'household_size_ward.xlsx'

for year, df in dfs_size.items():
    # print(year)
    lsoa_col = "geography_code"
    ward_col = "ward"
    geo_name = 'geography_name'
    num_col = "number"
    cat_col = "class_name"
    
    # get lsoa-level data
    is_lsoa = df["geography_type"].str.contains("lower layer", case=False, na=False)
    df_lsoa = df.loc[is_lsoa].copy()

    df_other = df.loc[~is_lsoa].copy()
    
    # map lsoa to 2021 ward
    df_ward_tmp = lsoa_to_ward(df_lsoa, year, lsoa_col, ward_col, ward_mapping)
    
    df_ward_tmp.drop(columns=['geography_code', geo_name], inplace=True)
    df_ward_tmp.rename(columns={'WD25CD': 'geography_code', 'ward': geo_name}, inplace=True)
    
    df_weight = df_ward_tmp.copy()

    df_weight[f"weighted_{num_col}"] = df_weight[num_col] * df_weight["weight"]

    global_keys = [cat_col, "census"]
    unique_keys = [geo_name] + global_keys

    # ward-level
    df_weight = (
        df_weight
        .groupby(unique_keys, as_index=False)
        .agg(
            **{
                num_col: (f"weighted_{num_col}", "sum"),
            }
        )
    )
    
    df_ward = df_weight.merge(df_ward_tmp.drop(columns=['weight', num_col]).drop_duplicates(), how='left', on=unique_keys)
    df_ward['geography_type'] = '2025 ward'
    
    df_new = pd.concat([df_other, df_ward], ignore_index=True)
   
    dfs_new.append(df_new)

df_size_ward = pd.concat(dfs_new, ignore_index=True)

total_row = "Total: All households"
pct_col = "pct"

df_size_ward.replace({'All categories: Household size': total_row}, inplace=True)

df_size_ward = df_size_ward[df_size_ward['class_name'] != '0 people in household'] # there are no empty households.

is_total = df_size_ward[cat_col].isin([total_row])
totals = df_size_ward[is_total][[geo_name, num_col, "census"]].rename(columns={num_col: 'total_households'})
df_size_ward = df_size_ward.merge(totals, on=[geo_name, "census"], how="left")
df_size_ward[pct_col] = (df_size_ward[num_col] / df_size_ward["total_households"] * 100).round(1)

geo_type_col = "geography_type"
is_district = df_size_ward[geo_type_col].str.contains("district|unitary|local authorit", case=False, na=False)
df_size_ward.loc[is_district, geo_type_col] = "local authority district"

df_size_ward.to_excel(save_path, index=False)
print(f"The final data has been saved at: [bold red]{save_path}[/bold red]")

Warning: 1 LSOA01CD values map to multiple WD25NM values. These will be split equally using weights.

,LSOA01CD,WD25CD,WD25NM
3584,E01010823,E05001347,City
3588,E01010823,E05001359,Manningham


The final data has been saved at: data/household_size_ward.xlsx

Have a look at the structure of our final dataset (long format):

In [22]:
df_size_ward.head()

,census,geography_code,geography_type,geography_name,class_order,class_name,class_type,number,total_households,pct
0,2001,E08000032,local authority district,Bradford,0,Total: All households,Household Size,180246.0,180246.0,100.0
1,2001,E08000032,local authority district,Bradford,1,1 person in household,Household Size,51991.0,180246.0,28.8
2,2001,E08000032,local authority district,Bradford,2,2 people in household,Household Size,56555.0,180246.0,31.4
3,2001,E08000032,local authority district,Bradford,3,3 people in household,Household Size,28359.0,180246.0,15.7
4,2001,E08000032,local authority district,Bradford,4,4 people in household,Household Size,24666.0,180246.0,13.7


In [23]:
save_path = data_dir / 'household_size_ward_wide.xlsx'

id_cols = [
    "geography_code",
    "geography_name",
    "geography_type",
    "census",
]

tmp = df_size_ward.loc[
    df_size_ward["class_name"] != "Total: All households"
].copy()

size_pct_wide = (
    tmp
    .pivot_table(
        index=id_cols,
        columns="class_name",
        values="pct",
        aggfunc="sum"
    )
    .add_suffix("_pct")
    .reset_index()
)

size_num_wide = (
    tmp
    .pivot_table(
        index=id_cols,
        columns="class_name",
        values="number",
        aggfunc="sum"
    )
    .add_suffix("_number")
    .reset_index()
)

size_wide = size_pct_wide.merge(
    size_num_wide,
    on=id_cols,
    how="left"
)

size_wide.columns.name = None

size_wide.to_excel(save_path, index=False)

print(f"Wide-format data saved at: [bold red]{save_path}[/bold red]")

Wide-format data saved at: data/household_size_ward_wide.xlsx

Have a look at the structure of our final dataset (wide format):

In [24]:
size_wide.head()

,geography_code,geography_name,geography_type,census,1 person in household_pct,2 people in household_pct,3 people in household_pct,4 people in household_pct,5 people in household_pct,6 people in household_pct,7 people in household_pct,8 or more people in household_pct,1 person in household_number,2 people in household_number,3 people in household_number,4 people in household_number,5 people in household_number,6 people in household_number,7 people in household_number,8 or more people in household_number
0,E05001341,Baildon,2025 ward,2001,27.5,37.5,15.3,14.8,3.9,0.7,0.2,0.1,1824.0,2488.0,1014.0,980.0,259.0,48.0,12.0,6.0
1,E05001341,Baildon,2025 ward,2011,31.3,36.4,15.5,12.4,3.4,0.9,0.1,0.1,2215.0,2575.0,1095.0,880.0,242.0,63.0,5.0,7.0
2,E05001341,Baildon,2025 ward,2021,31.8,38.4,14.6,11.6,2.8,0.7,0.1,0.1,2355.0,2840.0,1080.0,857.0,204.0,54.0,7.0,7.0
3,E05001342,Bingley,2025 ward,2001,29.5,35.0,16.1,14.1,3.9,1.0,0.1,0.2,1933.0,2292.0,1057.0,925.0,255.0,65.0,6.0,13.0
4,E05001342,Bingley,2025 ward,2011,31.8,36.6,14.8,12.4,3.4,0.7,0.2,0.1,2522.0,2904.0,1175.0,985.0,269.0,56.0,13.0,5.0
